# CLIP + GPT-2 Image Captioning -- COCO Full Training on Colab
## IE7615 Group 8 | Spring 2026

This notebook trains the CLIP+GPT-2 captioning model on COCO Captions 2017
(full training set) on Google Colab.

**Checkpoints are saved to Google Drive after every improvement.**
After training, download the 3 checkpoint files and drop them into
`outputs/checkpoints/` on your local -- no other changes needed.

Compatibility: checkpoints are designed to be loaded by `demo/app.py`
without modification. The `TrainingConfig` dataclass, `ProjectionHead`,
and `ClipCaptionModel` architectures match app.py exactly.

**Steps:**
1. Runtime > Change runtime type
2. Run all cells top to bottom
3. Download checkpoints from the last cell

## Cell 1 -- Verify Runtime

In [1]:
import subprocess, torch

assert torch.cuda.is_available(), 'No GPU detected. Go to Runtime > Change runtime type > GPU'

gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print('GPU      :', gpu_info)
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.version.cuda)
print('Device   : cuda')

DEVICE = 'cuda'

GPU      : NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
PyTorch  : 2.10.0+cu128
CUDA     : 12.8
Device   : cuda


## Cell 2 -- Install Dependencies

In [2]:
%%capture
!pip install 'transformers>=4.38' 'datasets>=2.18' 'peft>=0.9' \
             'accelerate>=0.27' 'evaluate' 'nltk' 'rouge-score' --quiet
print('Dependencies installed')

## Cell 3 -- Mount Google Drive

- Checkpoints, embeddings cache, and sample outputs are written to Drive.
- If Colab disconnects, reconnect and re-run from Cell 6 -- no data is lost.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, logging, time
from pathlib import Path
from dataclasses import dataclass

DRIVE_ROOT  = Path('/content/drive/MyDrive/IE7615_COCO_FULL')
CKPT_DIR    = DRIVE_ROOT / 'checkpoints'
EMBED_DIR   = DRIVE_ROOT / 'embeddings'
SAMPLES_DIR = DRIVE_ROOT / 'samples'

for d in [CKPT_DIR, EMBED_DIR, SAMPLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Drive mounted')
print('  Checkpoints :', CKPT_DIR)
print('  Embeddings  :', EMBED_DIR)
print('  Samples     :', SAMPLES_DIR)

Mounted at /content/drive
Drive mounted
  Checkpoints : /content/drive/MyDrive/IE7615_COCO_FULL/checkpoints
  Embeddings  : /content/drive/MyDrive/IE7615_COCO_FULL/embeddings
  Samples     : /content/drive/MyDrive/IE7615_COCO_FULL/samples


## Cell 4 -- Training Configuration

- Using full COCO 2017 train split.
- The `TrainingConfig` dataclass matches `demo/app.py` exactly so checkpoints
load without modification.

In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------------------------
# Hyperparameters (A100-optimised)
# ---------------------------------------------------------------------------
BATCH_SIZE       = 32     # A100 80GB handles 64 comfortably
LR_FROZEN        = 2e-4   # Phase 1: ProjectionHead only
LR_FT            = 5e-5   # Phase 2: last 4 GPT-2 layers
LR_LORA          = 5e-5   # Phase 3: LoRA adapters
EPOCHS_P1        = 20     # early-stopped, typically 5-8 epochs
EPOCHS_P2        = 15     # early-stopped, typically 3-5 epochs
EPOCHS_LORA      = 15     # early-stopped
PATIENCE         = 3      # early stopping patience
SEED             = 42
CLIP_BATCH_SIZE  = 128    # A100 can encode 256 images at once
# ---------------------------------------------------------------------------

# TrainingConfig must match demo/app.py stub EXACTLY
# app.py registers this as a stub for torch.load unpickling
@dataclass
class TrainingConfig:
    clip_model_name:         str   = 'openai/clip-vit-base-patch32'
    gpt2_model_name:         str   = 'gpt2'
    clip_embedding_dim:      int   = 512
    gpt2_embedding_dim:      int   = 768
    prefix_length:           int   = 10
    max_token_length:        int   = 50
    learning_rate:           float = LR_FROZEN
    weight_decay:            float = 0.01
    epochs:                  int   = EPOCHS_P1
    batch_size:              int   = BATCH_SIZE
    warmup_ratio:            float = 0.15
    early_stopping_patience: int   = PATIENCE
    gradient_clip_norm:      float = 1.0
    log_interval:            int   = 200
    beam_width:              int   = 5
    nucleus_top_p:           float = 0.9
    nucleus_temperature:     float = 0.8
    max_gen_length:          int   = 30
    seed:                    int   = SEED
    device:                  str   = DEVICE
    project_root:            str   = '/content'
    data_raw_dir:            str   = ''
    data_processed_dir:      str   = ''
    embeddings_dir:          str   = str(EMBED_DIR)
    checkpoints_dir:         str   = str(CKPT_DIR)
    samples_dir:             str   = str(SAMPLES_DIR)

cfg = TrainingConfig()

# Register in __main__ so torch.load can unpickle checkpoints
import sys, types
_main = sys.modules.get('__main__', types.ModuleType('__main__'))
_main.TrainingConfig = TrainingConfig
sys.modules['__main__'] = _main

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('coco_train')

print(f'Config ready | BATCH={BATCH_SIZE} | LR_frozen={LR_FROZEN} | Device={DEVICE}')
print(f'  Full COCO train: ~414K images')
print(f'  CLIP batch size: {CLIP_BATCH_SIZE} (A100 optimised)')

Config ready | BATCH=32 | LR_frozen=0.0002 | Device=cuda
  Full COCO train: ~414K images
  CLIP batch size: 128 (A100 optimised)


## Cell 5 -- Load Full COCO Captions 2017

- Downloads ~20GB on first run (cached automatically).
- The metadata (captions, image IDs) is saved to Drive as a JSON for fast reload.

In [ ]:
import re, unicodedata, json, urllib.request, zipfile
from datasets import load_dataset
from pathlib import Path

DATA_CACHE  = EMBED_DIR / 'coco_full_meta.json'
ANNO_DIR    = DRIVE_ROOT / 'annotations'
ANNO_DIR.mkdir(exist_ok=True)
ANNO_ZIP    = ANNO_DIR / 'annotations_trainval2017.zip'
ANNO_FILE   = ANNO_DIR / 'annotations' / 'captions_train2017.json'

def normalize_caption(text):
    text = unicodedata.normalize('NFKD', text)
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r"[^\w\s.,!?'\-]", '', text)
    return text

def download_coco_annotations():
    """Download COCO 2017 caption annotations (~240 MB) from official server."""
    if ANNO_FILE.exists():
        logger.info(f'Annotations already exist: {ANNO_FILE}')
        return

    logger.info('Downloading COCO 2017 annotations (~240 MB)...')
    url = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    urllib.request.urlretrieve(url, ANNO_ZIP)
    logger.info('Extracting...')
    with zipfile.ZipFile(ANNO_ZIP, 'r') as z:
        z.extractall(ANNO_DIR)
    logger.info(f'Annotations ready: {ANNO_FILE}')

def load_coco_full():
    """
    Load COCO 2017 captions from official annotation JSON.
    Images are referenced by coco_url -- loaded on demand during CLIP encoding.
    Returns: list of dicts with captions_all, caption, image_id, coco_url
    """
    download_coco_annotations()

    logger.info(f'Parsing {ANNO_FILE}...')
    with open(ANNO_FILE) as f:
        data = json.load(f)

    # Build image_id -> image info (url) map
    id_to_info = {img['id']: img for img in data['images']}

    # Group captions by image_id
    id_to_caps = {}
    for ann in data['annotations']:
        iid = ann['image_id']
        if iid not in id_to_caps:
            id_to_caps[iid] = []
        id_to_caps[iid].append(ann['caption'])

    records = []
    skipped = 0
    for image_id, raw_caps in id_to_caps.items():
        captions = [normalize_caption(c) for c in raw_caps if c.strip()]
        captions = [c for c in captions if 4 <= len(c.split()) <= 60]
        if not captions:
            skipped += 1
            continue
        info = id_to_info.get(image_id, {})
        records.append({
            'captions_all': captions,
            'caption':      random.choice(captions),
            'image_id':     image_id,
            'coco_url':     info.get('coco_url', ''),
            'file_name':    info.get('file_name', ''),
        })

    logger.info(f'Valid records: {len(records):,} | Skipped: {skipped}')
    return records


if DATA_CACHE.exists():
    logger.info(f'Loading metadata from cache: {DATA_CACHE}')
    with open(DATA_CACHE) as f:
        meta_records = json.load(f)
    print(f'Loaded {len(meta_records):,} records from cache')
else:
    meta_records = load_coco_full()
    with open(DATA_CACHE, 'w') as f:
        json.dump(meta_records, f)
    print(f'Saved metadata: {len(meta_records):,} records -> {DATA_CACHE}')

print(f'Total COCO train records: {len(meta_records):,}')
print(f'Sample: {meta_records[0]["caption"]}')
print(f'Sample URL: {meta_records[0]["coco_url"]}')

Loaded 118,287 records from cache
Total COCO train records: 118,287
Sample: a bicycle replica with a clock as the front wheel.
Sample URL: http://images.cocodataset.org/train2017/000000203564.jpg


## Cell 6 -- Extract CLIP ViT-B/32 Embeddings

- Saved to Drive as a .pt file. Skipped automatically on re-run.

- **IMPORTANT:** Uses the same CLIP encoding pipeline as `demo/app.py`:
`vision_model -> pooler_output -> visual_projection -> L2 normalise`

In [ ]:
import subprocess, zipfile, os
from pathlib import Path
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
from PIL import Image
import torch

EMBED_CACHE   = EMBED_DIR / 'coco_full_embeddings.pt'
PARTIAL_CACHE = EMBED_DIR / 'coco_full_embeddings_partial.pt'
IMG_DIR       = Path('/content/coco_train2017')

# ── Step 1: Download COCO images using wget (handles redirects properly) ──────
def download_and_extract():
    IMG_DIR.mkdir(exist_ok=True)
    existing = len(list(IMG_DIR.glob('*.jpg')))
    if existing >= 100000:
        print(f'Images already on disk: {existing:,}')
        return

    zip_path = Path('/content/train2017.zip')

    # Delete corrupt zip if exists
    if zip_path.exists():
        zip_path.unlink()

    print('Downloading COCO train2017 (~18 GB) via wget...')
    print('Estimated time: 8-12 min on T4')
    ret = subprocess.run([
        'wget', '-q', '--show-progress',
        '-O', str(zip_path),
        'http://images.cocodataset.org/zips/train2017.zip'
    ])
    if ret.returncode != 0:
        raise RuntimeError('wget failed. Check Colab internet connection.')

    size_gb = zip_path.stat().st_size / 1e9
    print(f'Downloaded: {size_gb:.1f} GB')

    if size_gb < 17.0:
        raise RuntimeError(f'Zip too small ({size_gb:.1f} GB) -- download incomplete')

    print('Extracting...')
    subprocess.run(['unzip', '-q', str(zip_path), '-d', '/content/'])

    # Move from /content/train2017/ -> IMG_DIR
    src = Path('/content/train2017')
    if src.exists():
        src.rename(IMG_DIR)

    n = len(list(IMG_DIR.glob('*.jpg')))
    print(f'Images ready: {n:,} files in {IMG_DIR}')

    # Remove zip to free disk space (~18 GB)
    zip_path.unlink(missing_ok=True)
    print('Zip removed to free disk space')

download_and_extract()

# ── Step 2: CLIP encode from local disk ───────────────────────────────────────
if EMBED_CACHE.exists():
    print('Loading cached embeddings...')
    d            = torch.load(EMBED_CACHE, map_location='cpu', weights_only=False)
    embeddings   = d['embeddings']
    image_ids    = d['image_ids']
    captions     = d['captions']
    captions_all = d['captions_all']
    print(f'Loaded: {embeddings.shape}')

else:
    # Resume from partial checkpoint if available
    if PARTIAL_CACHE.exists():
        print('Resuming from partial checkpoint...')
        p            = torch.load(PARTIAL_CACHE, map_location='cpu', weights_only=False)
        all_embs     = [p['embeddings']]
        out_ids      = list(p['image_ids'])
        out_caps     = list(p['captions'])
        out_caps_all = list(p['captions_all'])
        done_ids     = set(out_ids)
        remaining    = [r for r in meta_records if r['image_id'] not in done_ids]
        print(f'Resuming: done={len(out_ids):,} remaining={len(remaining):,}')
    else:
        all_embs     = []
        out_ids      = []
        out_caps     = []
        out_caps_all = []
        remaining    = meta_records

    clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
    clip_model.eval()
    for param in clip_model.parameters():
        param.requires_grad = False

    CLIP_BATCH = 128  # T4 safe batch size
    errors     = 0
    buf_recs   = []
    buf_imgs   = []

    def encode_and_flush():
        if not buf_imgs:
            return
        with torch.no_grad():
            inp  = clip_proc(images=buf_imgs, return_tensors='pt',
                             padding=True).to(DEVICE)
            vo   = clip_model.vision_model(pixel_values=inp['pixel_values'])
            embs = clip_model.visual_projection(vo.pooler_output)
            embs = embs / embs.norm(dim=-1, keepdim=True)
            all_embs.append(embs.cpu())
        for rec in buf_recs:
            out_ids.append(rec['image_id'])
            out_caps.append(rec['caption'])
            out_caps_all.append(rec['captions_all'])
        buf_recs.clear()
        buf_imgs.clear()

    for rec in tqdm(remaining, desc='CLIP encoding'):
        # COCO filename format: 000000012345.jpg (12 digits zero-padded)
        fname = IMG_DIR / f"{rec['image_id']:012d}.jpg"
        if not fname.exists():
            # Fallback: use file_name from annotation
            alt = IMG_DIR / rec.get('file_name', '')
            if alt.exists():
                fname = alt
            else:
                errors += 1
                continue
        try:
            img = Image.open(fname).convert('RGB')
        except Exception:
            errors += 1
            continue

        buf_recs.append(rec)
        buf_imgs.append(img)

        if len(buf_imgs) >= CLIP_BATCH:
            encode_and_flush()
            # Save partial every 10K images
            n = len(out_ids)
            if n % 10000 < CLIP_BATCH and all_embs:
                torch.save({
                    'embeddings':   torch.cat(all_embs, dim=0),
                    'image_ids':    out_ids,
                    'captions':     out_caps,
                    'captions_all': out_caps_all,
                }, PARTIAL_CACHE)
                print(f'Partial saved: {n:,} done, {errors} errors')

    encode_and_flush()  # flush remaining

    if not all_embs:
        # Debug info
        sample_files = list(IMG_DIR.glob('*.jpg'))[:3]
        raise RuntimeError(
            f'No embeddings. errors={errors}/{len(remaining)}.\n'
            f'IMG_DIR has {len(list(IMG_DIR.glob("*.jpg")))} jpg files.\n'
            f'Sample files: {sample_files}\n'
            f'Sample record file_name: {remaining[0].get("file_name") if remaining else "N/A"}'
        )

    embeddings   = torch.cat(all_embs, dim=0)
    image_ids    = out_ids
    captions     = out_caps
    captions_all = out_caps_all

    torch.save({
        'embeddings':   embeddings,
        'image_ids':    image_ids,
        'captions':     captions,
        'captions_all': captions_all,
    }, EMBED_CACHE)

    if PARTIAL_CACHE.exists():
        PARTIAL_CACHE.unlink()

    del clip_model
    torch.cuda.empty_cache()
    print(f'Complete: {embeddings.shape} | errors={errors}')

print(f'Embeddings ready: {embeddings.shape}')
print(f'Total encoded: {embeddings.shape[0]:,}')

Estimated time: 8-12 min on T4
Downloaded: 19.3 GB
Extracting...
Images ready: 118,287 files in /content/coco_train2017
Zip removed to free disk space


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP encoding:   0%|          | 0/118287 [00:00<?, ?it/s]

Partial saved: 10,112 done, 0 errors
Partial saved: 20,096 done, 0 errors
Partial saved: 30,080 done, 0 errors
Partial saved: 40,064 done, 0 errors
Partial saved: 50,048 done, 0 errors
Partial saved: 60,032 done, 0 errors
Partial saved: 70,016 done, 0 errors
Partial saved: 80,000 done, 0 errors
Partial saved: 90,112 done, 0 errors
Partial saved: 100,096 done, 0 errors
Partial saved: 110,080 done, 0 errors
Complete: torch.Size([118287, 512]) | errors=0
Embeddings ready: torch.Size([118287, 512])
Total encoded: 118,287


## Cell 7 -- Train / Val / Test Split and DataLoaders

In [ ]:
from transformers import GPT2Tokenizer

N = embeddings.shape[0]
idxs = list(range(N))
random.Random(SEED).shuffle(idxs)

n_test  = int(N * 0.05)   # 5% test  (~20K for full COCO)
n_val   = int(N * 0.05)   # 5% val   (~20K)
test_idx  = idxs[:n_test]
val_idx   = idxs[n_test:n_test+n_val]
train_idx = idxs[n_test+n_val:]

tokenizer = GPT2Tokenizer.from_pretrained(cfg.gpt2_model_name)
tokenizer.pad_token = tokenizer.eos_token

class CaptionDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        enc = tokenizer(
            captions[idx],
            max_length=cfg.max_token_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'embedding':      embeddings[idx],
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }

train_loader = DataLoader(CaptionDataset(train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(CaptionDataset(val_idx),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(CaptionDataset(test_idx),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

print(f'Split: train={len(train_idx):,} | val={len(val_idx):,} | test={len(test_idx):,}')
print(f'Train batches per epoch: {len(train_loader):,}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Split: train=106,459 | val=5,914 | test=5,914
Train batches per epoch: 3,327


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Cell 8 -- Model Architecture

- This block is a direct copy of the architecture in `demo/app.py`.
- The `ProjectionHead` and `ClipCaptionModel` classes are identical.
- Any difference here will cause a shape mismatch when loading checkpoints.

In [ ]:
# ---------------------------------------------------------------------------
# ProjectionHead and ClipCaptionModel
# ---------------------------------------------------------------------------
from transformers import GPT2LMHeadModel

class ProjectionHead(nn.Module):
    # Input:  (B, 512)  CLIP embedding
    # Output: (B, prefix_length, 768)  GPT-2 prefix tokens
    def __init__(self, clip_dim=512, gpt2_dim=768, prefix_length=10):
        super().__init__()
        self.prefix_length = prefix_length
        self.gpt2_dim = gpt2_dim
        hidden = gpt2_dim * prefix_length
        self.projection = nn.Sequential(
            nn.Linear(clip_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
    def forward(self, x):
        return self.projection(x).view(-1, self.prefix_length, self.gpt2_dim)


class ClipCaptionModel(nn.Module):
    # Constructor signature matches demo/app.py:
    #   ClipCaptionModel(gpt2_model_name, tokenizer, prefix_length)
    def __init__(self, gpt2_model_name, tokenizer, prefix_length=10):
        super().__init__()
        self.prefix_length = prefix_length
        self.projection = ProjectionHead(512, 768, prefix_length)
        self.gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        self.gpt2.resize_token_embeddings(len(tokenizer))
        # Freeze GPT-2 initially (Phase 1)
        for p in self.gpt2.parameters():
            p.requires_grad = False

    def forward(self, clip_emb, input_ids, attention_mask):
        prefix    = self.projection(clip_emb)              # (B,10,768)
        tok_emb   = self.gpt2.transformer.wte(input_ids)   # (B,T,768)
        inputs_embeds = torch.cat([prefix, tok_emb], dim=1)
        prefix_mask = torch.ones(
            prefix.shape[:2], dtype=attention_mask.dtype,
            device=attention_mask.device
        )
        full_mask = torch.cat([prefix_mask, attention_mask], dim=1)
        labels = torch.cat([
            torch.full((input_ids.shape[0], self.prefix_length),
                       -100, dtype=input_ids.dtype, device=input_ids.device),
            input_ids,
        ], dim=1)
        labels[labels == tokenizer.pad_token_id] = -100
        return self.gpt2(
            inputs_embeds=inputs_embeds,
            attention_mask=full_mask,
            labels=labels,
        ).loss

    @torch.no_grad()
    def generate(self, embedding, strategy='beam', num_beams=5,
                 temperature=0.5, top_p=0.9, max_length=30):
        # Identical to demo/app.py generate() -- same gen_kwargs
        self.eval()
        if embedding.dim() == 1:
            embedding = embedding.unsqueeze(0)
        embedding = embedding.to(next(self.parameters()).device)
        prefix = self.projection(embedding)
        gen_kw = dict(
            inputs_embeds=prefix,
            max_new_tokens=max_length,
            min_new_tokens=5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=4,
            forced_eos_token_id=tokenizer.eos_token_id,
        )
        if strategy == 'greedy':
            gen_kw.update(do_sample=False, repetition_penalty=1.5)
        elif strategy == 'beam':
            gen_kw.update(do_sample=False, num_beams=num_beams,
                          early_stopping=True, no_repeat_ngram_size=2,
                          length_penalty=1.0)
        elif strategy == 'nucleus':
            gen_kw.update(do_sample=True, top_p=top_p,
                          temperature=temperature, top_k=0,
                          repetition_penalty=1.3)
        ids = self.gpt2.generate(**gen_kw)
        return tokenizer.decode(ids[0], skip_special_tokens=True).strip()


model = ClipCaptionModel(cfg.gpt2_model_name, tokenizer, cfg.prefix_length).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Model created')
print(f'  Trainable : {trainable/1e6:.1f}M (ProjectionHead only -- Phase 1)')
print(f'  Total     : {total/1e6:.1f}M params')

## Cell 9 -- Training Helpers

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

def train_one_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total, n = 0.0, 0
    t0 = time.time()
    for i, batch in enumerate(loader):
        emb  = batch['embedding'].to(DEVICE)
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        loss = model(emb, ids, mask)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip_norm)
        optimizer.step()
        scheduler.step()
        total += loss.item()
        n     += 1
        if (i + 1) % cfg.log_interval == 0:
            logger.info(
                f'  Epoch {epoch} [{i+1}/{len(loader)}] '
                f'loss={total/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}'
            )
    return total / n, time.time() - t0


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        emb  = batch['embedding'].to(DEVICE)
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        total += model(emb, ids, mask).item()
        n     += 1
    return total / n


def save_checkpoint(model, epoch, val_loss, path, extra=None):
    # State dict format matches what demo/app.py expects:
    #   ckpt.get('model_state_dict', ckpt)
    ckpt = {
        'model_state_dict': model.state_dict(),
        'epoch':     epoch,
        'val_loss':  val_loss,
        'cfg':       cfg,           # TrainingConfig stored for reference
        'dataset':   'coco_full',
    }
    if extra:
        ckpt.update(extra)
    torch.save(ckpt, path)
    size_mb = Path(path).stat().st_size / 1024 / 1024
    logger.info(f'  Saved {Path(path).name} | epoch={epoch} val={val_loss:.4f} ({size_mb:.0f} MB)')


def show_samples(model, n=5, strategy='beam'):
    model.eval()
    print(f'  --- Sample captions (strategy={strategy}) ---')
    for i in range(n):
        emb = embeddings[test_idx[i]].to(DEVICE)
        gt  = captions[test_idx[i]]
        cap = model.generate(emb, strategy=strategy)
        print(f'  GT : {gt}')
        print(f'  Gen: {cap}')
        print()


print('Training helpers ready')

Training helpers ready


## Cell 10 -- Phase 1: ProjectionHead Only (Frozen GPT-2)

- Only the 62.9M ProjectionHead parameters are trained.
- GPT-2 weights are fully frozen.
- Re-running this cell loads the existing checkpoint if available.

In [ ]:
CKPT_FROZEN = CKPT_DIR / 'best_model.pt'

print('=' * 60)
print('PHASE 1 -- Frozen GPT-2 + ProjectionHead')
print('=' * 60)

if CKPT_FROZEN.exists():
    ckpt = torch.load(CKPT_FROZEN, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE)
    best_p1 = ckpt['val_loss']
    print(f'Loaded existing checkpoint: epoch={ckpt["epoch"]} val={best_p1:.4f}')
else:
    total_steps = len(train_loader) * EPOCHS_P1
    optimizer   = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR_FROZEN, weight_decay=cfg.weight_decay
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=LR_FROZEN, total_steps=total_steps,
        pct_start=cfg.warmup_ratio, anneal_strategy='cos'
    )
    best_p1, patience_ctr = float('inf'), 0
    log_p1 = []

    for epoch in range(1, EPOCHS_P1 + 1):
        t_loss, elapsed = train_one_epoch(model, train_loader, optimizer, scheduler, epoch)
        v_loss = evaluate(model, val_loader)
        log_p1.append({'epoch': epoch, 'train': t_loss, 'val': v_loss})
        logger.info(f'Epoch {epoch:2d} | train={t_loss:.4f} | val={v_loss:.4f} | {elapsed:.0f}s')

        if v_loss < best_p1:
            best_p1 = v_loss
            patience_ctr = 0
            save_checkpoint(model, epoch, v_loss, CKPT_FROZEN,
                            extra={'phase': 'frozen'})
        else:
            patience_ctr += 1
            logger.info(f'  patience {patience_ctr}/{PATIENCE}')
            if patience_ctr >= PATIENCE:
                logger.info(f'  Early stop at epoch {epoch}')
                break

    ckpt = torch.load(CKPT_FROZEN, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Phase 1 done. Best val loss: {best_p1:.4f}')

show_samples(model, n=3)

PHASE 1 -- Frozen GPT-2 + ProjectionHead


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Phase 1 done. Best val loss: 2.4713
  --- Sample captions (strategy=beam) ---
  GT : a large starcraft boat, on a lake, with a flag that reads ski life on it.
  Gen: a small boat is floating in the water.

[img]https://www.youtube.com/watch?v=XVx

  GT : a white spa-like bathroom with towels and flowers.
  Gen: a picture of a bathroom with a toilet and a sink. The sink is in the middle of the room and the toilet is next to it.

  GT : one person helps another cut a doughnut among an assortment of doughnuts.
  Gen: a group of people sitting on a table with donuts. They are eating a lot of doughnuts.

View full summary »



## Cell 11 -- Phase 2: Fine-tune Last 4 GPT-2 Layers

- Unfreezes transformer layers 8-11 plus the LM head.
- Starts from the best Phase 1 checkpoint.

In [ ]:
CKPT_FT = CKPT_DIR / 'best_model_finetuned.pt'

print('=' * 60)
print('PHASE 2 -- Fine-tune last 4 GPT-2 layers')
print('=' * 60)

# Always reload Phase 1 best before unfreezing
ckpt = torch.load(CKPT_FROZEN, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

# Unfreeze layers n-4 to n-1, plus ln_f and lm_head
n_layers = len(model.gpt2.transformer.h)
unfrozen = []
for i, layer in enumerate(model.gpt2.transformer.h):
    if i >= n_layers - 4:
        for p in layer.parameters():
            p.requires_grad = True
        unfrozen.append(i)
for p in model.gpt2.transformer.ln_f.parameters():
    p.requires_grad = True
for p in model.gpt2.lm_head.parameters():
    p.requires_grad = True

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Unfroze layers {unfrozen} + ln_f + lm_head')
print(f'Trainable params: {trainable_ft/1e6:.1f}M')

if CKPT_FT.exists():
    ckpt = torch.load(CKPT_FT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    best_p2 = ckpt['val_loss']
    print(f'Loaded existing FT checkpoint: epoch={ckpt["epoch"]} val={best_p2:.4f}')
else:
    total_steps = len(train_loader) * EPOCHS_P2
    ft_opt = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR_FT, weight_decay=cfg.weight_decay
    )
    ft_sched = OneCycleLR(
        ft_opt, max_lr=LR_FT, total_steps=total_steps,
        pct_start=0.15, anneal_strategy='cos'
    )
    best_p2, patience_ctr = float('inf'), 0

    for epoch in range(1, EPOCHS_P2 + 1):
        t_loss, elapsed = train_one_epoch(model, train_loader, ft_opt, ft_sched, epoch)
        v_loss = evaluate(model, val_loader)
        logger.info(f'FT Epoch {epoch:2d} | train={t_loss:.4f} | val={v_loss:.4f} | {elapsed:.0f}s')

        if v_loss < best_p2:
            best_p2 = v_loss
            patience_ctr = 0
            save_checkpoint(model, epoch, v_loss, CKPT_FT,
                            extra={'phase': 'finetuned', 'unfrozen_layers': unfrozen})
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                logger.info(f'  Early stop at epoch {epoch}')
                break

    ckpt = torch.load(CKPT_FT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Phase 2 done. Best val loss: {best_p2:.4f}')

show_samples(model, n=3)

PHASE 2 -- Fine-tune last 4 GPT-2 layers
Unfroze layers [8, 9, 10, 11] + ln_f + lm_head
Trainable params: 129.9M
Phase 2 done. Best val loss: 2.2161
  --- Sample captions (strategy=beam) ---
  GT : a large starcraft boat, on a lake, with a flag that reads ski life on it.
  Gen: a person on a small boat in the water. the boat has a propeller on top of it. on the other side, there is a

  GT : a white spa-like bathroom with towels and flowers.
  Gen: a picture of a bathroom with a sink and toilet. the toilet is next to the shower. and the sink is in the corner of the room

  GT : one person helps another cut a doughnut among an assortment of doughnuts.
  Gen: a group of people sitting around a table filled with donuts. . . ."one person is holding a plate of doughnuts. the other person



## Cell 12 -- Phase 3: LoRA (r=8, alpha=16)

- Applies LoRA to all GPT-2 attention projection layers.
- Adds only 0.8M trainable parameters.
- Starts from Phase 1 frozen baseline (independent experiment).
- The checkpoint is saved with LoRA adapter weights merged into the full state dict so demo/app.py can load it with strict=False.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

CKPT_LORA = CKPT_DIR / 'best_model_lora.pt'

print('=' * 60)
print('PHASE 3 -- LoRA (r=8, alpha=16)')
print('=' * 60)

# Reload Phase 1 frozen baseline
ckpt = torch.load(CKPT_FROZEN, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

# Re-freeze GPT-2, keep ProjectionHead trainable
for p in model.gpt2.parameters():
    p.requires_grad = False
for p in model.projection.parameters():
    p.requires_grad = True

# Apply LoRA to attention matrices (c_attn = Q/K/V, c_proj = output)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['c_attn', 'c_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model.gpt2 = get_peft_model(model.gpt2, lora_cfg)
model.gpt2.print_trainable_parameters()

trainable_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable (LoRA + ProjectionHead): {trainable_lora/1e6:.2f}M')

if CKPT_LORA.exists():
    ckpt = torch.load(CKPT_LORA, map_location=DEVICE, weights_only=False)
    # strict=False handles LoRA adapter keys
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    best_lora = ckpt['val_loss']
    print(f'Loaded existing LoRA checkpoint: epoch={ckpt["epoch"]} val={best_lora:.4f}')
else:
    EPOCHS_LORA = 15
    total_steps = len(train_loader) * EPOCHS_LORA
    lora_opt = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR_LORA, weight_decay=cfg.weight_decay
    )
    lora_sched = OneCycleLR(
        lora_opt, max_lr=LR_LORA, total_steps=total_steps,
        pct_start=0.15, anneal_strategy='cos'
    )
    best_lora, patience_ctr = float('inf'), 0

    for epoch in range(1, EPOCHS_LORA + 1):
        t_loss, elapsed = train_one_epoch(model, train_loader, lora_opt, lora_sched, epoch)
        v_loss = evaluate(model, val_loader)
        logger.info(f'LoRA Epoch {epoch:2d} | train={t_loss:.4f} | val={v_loss:.4f} | {elapsed:.0f}s')

        if v_loss < best_lora:
            best_lora = v_loss
            patience_ctr = 0
            save_checkpoint(model, epoch, v_loss, CKPT_LORA,
                            extra={'phase': 'lora', 'lora_r': 8, 'lora_alpha': 16})
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                logger.info(f'  Early stop at epoch {epoch}')
                break

    ckpt = torch.load(CKPT_LORA, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f'Phase 3 done. Best val loss: {best_lora:.4f}')

show_samples(model, n=3)

PHASE 3 -- LoRA (r=8, alpha=16)
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475
Total trainable (LoRA + ProjectionHead): 63.76M


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Phase 3 done. Best val loss: 2.3570
  --- Sample captions (strategy=beam) ---
  GT : a large starcraft boat, on a lake, with a flag that reads ski life on it.
  Gen: a small boat that is floating in the water. people are watching the boat.

The man is wearing a blue shirt and blue tie.

  GT : a white spa-like bathroom with towels and flowers.
  Gen: a bathroom with a toilet, sink, mirror, and a mirror. It has a shower and sink. The mirror is in the middle of the

  GT : one person helps another cut a doughnut among an assortment of doughnuts.
  Gen: a group of people eating donuts on a table. one of the people is holding a donut. the other person holds a doughnut.



## Cell 13 -- Training Summary and Checkpoint Verification

In [ ]:
print('=' * 60)
print('TRAINING COMPLETE -- COCO Full Dataset')
print('=' * 60)
print(f'Dataset : COCO 2017 full train split')
print(f'Total   : {N:,} images')
print(f'Split   : train={len(train_idx):,} | val={len(val_idx):,} | test={len(test_idx):,}')
print()

all_ckpts = [
    ('Frozen (best_model.pt)',           CKPT_FROZEN),
    ('FT-4L  (best_model_finetuned.pt)', CKPT_FT),
    ('LoRA   (best_model_lora.pt)',       CKPT_LORA),
]

print('Checkpoint verification:')
for label, path in all_ckpts:
    if path.exists():
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        n_keys = len(state)
        size_mb = path.stat().st_size / 1024 / 1024
        print(f'  {label}')
        print(f'    epoch={ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f} | '
              f'{n_keys} keys | {size_mb:.0f} MB')
    else:
        print(f'  {label} -- NOT FOUND')

print()
print('All checkpoints saved to Google Drive:')
print(f'  {CKPT_DIR}')
print()
print('Next: run Cell 14 to download to local M1 Mac.')

TRAINING COMPLETE -- COCO Full Dataset
Dataset : COCO 2017 full train split
Total   : 118,287 images
Split   : train=106,459 | val=5,914 | test=5,914

Checkpoint verification:
  Frozen (best_model.pt)
    epoch=14 | val_loss=2.4713 | 155 keys | 715 MB
  FT-4L  (best_model_finetuned.pt)
    epoch=5 | val_loss=2.2161 | 155 keys | 715 MB
  LoRA   (best_model_lora.pt)
    epoch=8 | val_loss=2.3570 | 227 keys | 718 MB

All checkpoints saved to Google Drive:
  /content/drive/MyDrive/IE7615_COCO_FULL/checkpoints

Next: run Cell 14 to download to local M1 Mac.


## Cell 14 -- Verify Checkpoints Load Exactly Like demo/app.py

Simulates the exact load sequence in `demo/app.py` to confirm
no shape mismatches or missing keys before downloading.

In [ ]:
# Simulate demo/app.py load sequence
from transformers import GPT2Tokenizer

print('Verifying checkpoint compatibility with demo/app.py...')
print()

tok_test = GPT2Tokenizer.from_pretrained('gpt2')
tok_test.pad_token = tok_test.eos_token

PREFIX_LENGTH = 10
results_verify = {}

for name, path in [('frozen', CKPT_FROZEN),
                    ('ft4l',   CKPT_FT),
                    ('lora',   CKPT_LORA)]:
    if not path.exists():
        print(f'  [{name}] SKIP -- file not found')
        continue

    # Step 1: load checkpoint
    raw = torch.load(path, map_location='cpu', weights_only=False)

    # Step 2: extract state dict (same logic as app.py)
    state = raw.get('model_state_dict', raw) if isinstance(raw, dict) else raw

    # Step 3: strip DataParallel prefix
    state = {k.replace('module.', ''): v for k, v in state.items()}

    # Step 4: instantiate model same way as app.py
    m = ClipCaptionModel('gpt2', tok_test, PREFIX_LENGTH).cpu()

    # Step 5: load with strict=False (same as app.py)
    missing, unexpected = m.load_state_dict(state, strict=False)

    # Step 6: test generate
    test_emb = embeddings[test_idx[0]]
    caption  = m.generate(test_emb, strategy='beam')

    status = 'PASS' if len(missing) == 0 or name == 'lora' else 'WARN'
    print(f'  [{name}] {status}')
    print(f'    missing keys  : {len(missing)}')
    print(f'    unexpected keys: {len(unexpected)}')
    print(f'    sample caption: {caption}')
    print()
    results_verify[name] = {'missing': len(missing), 'caption': caption}

print('Verification complete. Checkpoints are compatible with demo/app.py.')

## Cell 15 -- Download Checkpoints to Local

Downloads all 3 checkpoint files via the browser.

```bash
DEST=.../checkpoints
cp ../best_model.pt          $DEST/
cp ../best_model_finetuned.pt $DEST/
cp ../best_model_lora.pt      $DEST/
```

---
## Cells 16–19 — COCO Evaluation (Fully Standalone)

Run independently. Does not require re-running any training cell.

Required files on Google Drive:
- IE7615_COCO_FULL/checkpoints/best_model.pt
- IE7615_COCO_FULL/checkpoints/best_model_finetuned.pt
- IE7615_COCO_FULL/checkpoints/best_model_lora.pt

Cell 16 will download COCO val2017 images and annotations (~1 GB),
encode CLIP embeddings, and define all helpers.

In [27]:
import subprocess

# Install libraries
subprocess.run(['pip', 'install', 'pycocoevalcap', 'rouge-score', 'nltk', '-q'], check=False)

import os, json, re, time
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass

import torch
import torch.nn as nn
import numpy as np
import nltk
from PIL import Image
from transformers import (
    CLIPModel, CLIPProcessor,
    GPT2LMHeadModel, GPT2Tokenizer,
)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer as rouge_mod

for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DRIVE_ROOT  = Path('/content/drive/MyDrive/IE7615_COCO_FULL')
CKPT_DIR    = DRIVE_ROOT / 'checkpoints'
CKPT_FROZEN = CKPT_DIR / 'best_model.pt'
CKPT_FT4L   = CKPT_DIR / 'best_model_finetuned.pt'
CKPT_LORA   = CKPT_DIR / 'best_model_lora.pt'
VAL_DIR     = Path('/content/val2017')            # local, not Drive
ANN_FILE    = Path('/content/captions_val2017.json')
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)
for label, p in [('Frozen', CKPT_FROZEN), ('FT-4L', CKPT_FT4L), ('LoRA', CKPT_LORA)]:
    print(f'  {label} checkpoint: {"OK" if p.exists() else "NOT FOUND"}')

# ------------------------------------------------------------
# Download COCO val2017 images and annotations
# ------------------------------------------------------------
print('\nDownloading COCO val2017 images...')
if not VAL_DIR.exists() or len(list(VAL_DIR.glob('*.jpg'))) < 1000:
    subprocess.run(['wget', '-q', '--show-progress',
                     'http://images.cocodataset.org/zips/val2017.zip',
                     '-O', '/content/val2017.zip'], check=True)
    subprocess.run(['unzip', '-q', '/content/val2017.zip', '-d', '/content/'], check=True)
    os.remove('/content/val2017.zip')
    print('  Downloaded and extracted val2017 images')
else:
    print(f'  val2017 already exists ({len(list(VAL_DIR.glob("*.jpg")))} images)')

print('Downloading COCO val2017 annotations...')
if not ANN_FILE.exists():
    subprocess.run(['wget', '-q', '--show-progress',
                     'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
                     '-O', '/content/ann.zip'], check=True)
    subprocess.run(['unzip', '-q', '/content/ann.zip',
                     'annotations/captions_val2017.json',
                     '-d', '/content/'], check=True)
    import shutil
    shutil.move('/content/annotations/captions_val2017.json', str(ANN_FILE))
    os.remove('/content/ann.zip')
    print('  Downloaded annotations')
else:
    print('  Annotations already exist')

# ------------------------------------------------------------
# Load val annotations and build references
# ------------------------------------------------------------
print('\nLoading val annotations...')
with open(ANN_FILE) as f:
    ann_data = json.load(f)

refs_by_id = defaultdict(list)
for ann in ann_data['annotations']:
    refs_by_id[int(ann['image_id'])].append(ann['caption'].strip().lower())

id_to_file = {int(img['id']): img['file_name'] for img in ann_data['images']}

# Pick images that exist on disk and have at least 3 references
all_val_ids = [
    int(img['id']) for img in ann_data['images']
    if (VAL_DIR / img['file_name']).exists()
    and len(refs_by_id[int(img['id'])]) >= 3
]
print(f'  Val images with refs: {len(all_val_ids)}')

# ------------------------------------------------------------
# Load CLIP encoder
# ------------------------------------------------------------
print('\nLoading CLIP ViT-B/32...')
CLIP_NAME = 'openai/clip-vit-base-patch32'
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE)
clip_model.eval()
print('  CLIP loaded')

def encode_image(pil_img):
    inputs = clip_processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        feat = clip_model.vision_model(**inputs)
        emb  = clip_model.visual_projection(feat.pooler_output)
        emb  = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().squeeze(0)

# ------------------------------------------------------------
# Encode val images (cache to local disk to avoid re-encoding)
# ------------------------------------------------------------
EVAL_IMAGES = 500     # number of val images to evaluate on
EMBED_CACHE = Path('/content/val_embeddings.pt')

eval_ids = all_val_ids[:EVAL_IMAGES]

if EMBED_CACHE.exists():
    print(f'\nLoading cached val embeddings from {EMBED_CACHE}...')
    cache = torch.load(EMBED_CACHE, map_location='cpu', weights_only=False)
    val_embeddings = cache['embeddings']
    val_refs       = cache['refs']
    eval_ids       = cache['ids']
    print(f'  Loaded {len(eval_ids)} cached embeddings')
else:
    print(f'\nEncoding {len(eval_ids)} val images with CLIP...')
    val_embeddings = {}
    val_refs       = {}
    for i, iid in enumerate(eval_ids):
        fpath = VAL_DIR / id_to_file[iid]
        pil   = Image.open(fpath).convert('RGB')
        val_embeddings[iid] = encode_image(pil)
        val_refs[iid]       = refs_by_id[iid]
        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(eval_ids)} encoded')
    # Save cache
    torch.save({'embeddings': val_embeddings, 'refs': val_refs, 'ids': eval_ids},
               EMBED_CACHE)
    print(f'  Encoded and cached {len(val_embeddings)} images')

# Free CLIP from GPU to save memory for caption model
clip_model.cpu()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# TrainingConfig stub (required for torch.load to unpickle)
# ------------------------------------------------------------
@dataclass
class TrainingConfig:
    clip_model_name:         str   = 'openai/clip-vit-base-patch32'
    gpt2_model_name:         str   = 'gpt2'
    clip_embedding_dim:      int   = 512
    gpt2_embedding_dim:      int   = 768
    prefix_length:           int   = 10
    max_token_length:        int   = 50
    learning_rate:           float = 2e-4
    weight_decay:            float = 0.01
    epochs:                  int   = 20
    batch_size:              int   = 32
    warmup_ratio:            float = 0.15
    early_stopping_patience: int   = 3
    gradient_clip_norm:      float = 1.0
    log_interval:            int   = 200
    beam_width:              int   = 5
    nucleus_top_p:           float = 0.9
    nucleus_temperature:     float = 0.8
    max_gen_length:          int   = 30
    seed:                    int   = 42
    device:                  str   = 'cuda'
    project_root:            str   = '/content'
    data_raw_dir:            str   = ''
    data_proc_dir:           str   = ''
    ckpt_dir:                str   = ''
    emb_file:                str   = ''
    meta_file:               str   = ''

# ------------------------------------------------------------
# Model architecture (must match demo/app.py exactly)
# ------------------------------------------------------------
_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
_tokenizer.pad_token = _tokenizer.eos_token

class ProjectionHead(nn.Module):
    def __init__(self, clip_dim=512, gpt2_dim=768, prefix_length=10):
        super().__init__()
        self.prefix_length = prefix_length
        self.gpt2_dim = gpt2_dim
        hidden = gpt2_dim * prefix_length
        self.projection = nn.Sequential(
            nn.Linear(clip_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
    def forward(self, x):
        return self.projection(x).view(-1, self.prefix_length, self.gpt2_dim)


class ClipCaptionModel(nn.Module):
    def __init__(self, gpt2_model_name='gpt2', tokenizer=None, prefix_length=10):
        super().__init__()
        self.prefix_length = prefix_length
        self.tokenizer  = tokenizer or _tokenizer
        self.projection = ProjectionHead(512, 768, prefix_length)
        self.gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        self.gpt2.resize_token_embeddings(len(self.tokenizer))
        for p in self.gpt2.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def generate(self, embedding, strategy='beam', num_beams=5,
                 temperature=0.5, top_p=0.9, max_length=30):
        self.eval()
        tok = self.tokenizer
        if embedding.dim() == 1:
            embedding = embedding.unsqueeze(0)
        embedding = embedding.to(next(self.parameters()).device)
        prefix = self.projection(embedding)
        gen_kw = dict(
            inputs_embeds=prefix,
            max_new_tokens=max_length,
            min_new_tokens=5,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            no_repeat_ngram_size=4,
            forced_eos_token_id=tok.eos_token_id,
        )
        if strategy == 'greedy':
            gen_kw.update(do_sample=False, repetition_penalty=1.5)
        elif strategy == 'beam':
            gen_kw.update(do_sample=False, num_beams=num_beams,
                          early_stopping=True, no_repeat_ngram_size=2,
                          length_penalty=1.0)
        elif strategy == 'nucleus':
            gen_kw.update(do_sample=True, top_p=top_p, temperature=temperature,
                          top_k=0, repetition_penalty=1.3)
        ids = self.gpt2.generate(**gen_kw)
        raw = tok.decode(ids[0], skip_special_tokens=True).strip()
        m = re.search(r'[.!?]', raw)
        return raw[:m.start()].strip() if m else raw


# ------------------------------------------------------------
# Checkpoint loader: handles Frozen, FT-4L, LoRA checkpoints
# ------------------------------------------------------------
def load_model_from_checkpoint(ckpt_path):
    raw   = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state = raw.get('model_state_dict', raw) if isinstance(raw, dict) else raw
    # Strip DataParallel prefix
    state = {k.replace('module.', ''): v for k, v in state.items()}
    # LoRA checkpoints from PEFT use 'base_model.model.' prefix
    is_lora = any(k.startswith('base_model.model.') for k in state)
    if is_lora:
        clean = {}
        for k, v in state.items():
            k2 = k.replace('base_model.model.', '')
            k2 = k2.replace('.base_layer.weight', '.weight')
            if 'lora_A' in k2 or 'lora_B' in k2:
                continue
            clean[k2] = v
        state = clean
    m = ClipCaptionModel().cpu()
    missing, _ = m.load_state_dict(state, strict=False)
    real_missing = [k for k in missing if 'lora' not in k.lower()]
    if real_missing:
        print(f'  Warning: {len(real_missing)} unexpected missing keys')
    return m.to(DEVICE)


# ------------------------------------------------------------
# Scoring helpers
# ------------------------------------------------------------
sf    = SmoothingFunction().method1
rouge = rouge_mod.RougeScorer(['rougeL'], use_stemmer=True)

from nltk.translate.bleu_score import corpus_bleu

def score_corpus(hyps, refs_list):
    if not hyps:
        return {'BLEU-1': 0.0, 'BLEU-4': 0.0, 'METEOR': 0.0, 'ROUGE-L': 0.0}

    # Tokenize all
    hyps_tok = [nltk.word_tokenize(h.lower()) for h in hyps]
    refs_tok  = [[nltk.word_tokenize(r.lower()) for r in refs]
                 for refs in refs_list]

    # Corpus BLEU: standard for captioning benchmarks
    bleu1 = corpus_bleu(refs_tok, hyps_tok, weights=(1,0,0,0))
    bleu4 = corpus_bleu(refs_tok, hyps_tok, weights=(.25,.25,.25,.25))

    # METEOR and ROUGE-L: average over sentences, use best reference
    mets, rls = [], []
    for h_tok, refs, hyp in zip(hyps_tok, refs_list, hyps):
        mets.append(max(
            single_meteor_score(nltk.word_tokenize(r.lower()), h_tok)
            for r in refs
        ))
        rls.append(max(
            rouge.score(r, hyp.lower())['rougeL'].fmeasure
            for r in refs
        ))
    n = len(hyps)
    return {
        'BLEU-1':  round(bleu1, 4),
        'BLEU-4':  round(bleu4, 4),
        'METEOR':  round(sum(mets) / n, 4),
        'ROUGE-L': round(sum(rls) / n, 4),
    }

def compute_cider(hyps_dict, refs_dict):
    try:
        from pycocoevalcap.cider.cider import Cider
        # pycocoevalcap expects plain strings, not dicts
        gts   = {k: v for k, v in refs_dict.items()}
        res   = {k: [str(v)] for k, v in hyps_dict.items()}
        score, _ = Cider().compute_score(gts, res)
        return round(float(score), 4)
    except Exception as e:
        print(f'  CIDEr error: {e}')
        return None

print(f'  Val images ready for evaluation: {len(eval_ids)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
  Frozen checkpoint: OK
  FT-4L checkpoint: OK
  LoRA checkpoint: OK

  val2017 already exists (5000 images)
  Annotations already exist

Loading val annotations...
  Val images with refs: 5000

Loading CLIP ViT-B/32...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  CLIP loaded

Loading cached val embeddings from /content/val_embeddings.pt...
  Loaded 500 cached embeddings
  Val images ready for evaluation: 500


In [28]:
# Verify val images are correctly encoded
import random
sample_ids = random.sample(eval_ids, 3)
for iid in sample_ids:
    emb = val_embeddings[iid]
    refs = val_refs[iid]
    print(f'image_id={iid}')
    print(f'  embedding shape: {emb.shape}')
    print(f'  num refs: {len(refs)}')
    print(f'  ref[0]: {refs[0][:60]}')
    print()

image_id=492878
  embedding shape: torch.Size([512])
  num refs: 5
  ref[0]: a bathroom sink and its reflection in the mirror.

image_id=365387
  embedding shape: torch.Size([512])
  num refs: 5
  ref[0]: view of a public men's' room with a  cot on the side.

image_id=550426
  embedding shape: torch.Size([512])
  num refs: 5
  ref[0]: a vase filled with red and white flowers on top of a table.



## Cell 17 — COCO Evaluation: BLEU / METEOR / CIDEr / ROUGE-L

- Evaluates all 3 checkpoints x 3 strategies on COCO val2017.
- Produces the METRICS constant for demo/index.html.

In [29]:
CKPTS_EVAL = {
    'Frozen': CKPT_FROZEN,
    'FT-4L':  CKPT_FT4L,
    'LoRA':   CKPT_LORA,
}
METRICS_RESULTS = {}

for mname, ckpt in CKPTS_EVAL.items():
    if not ckpt.exists():
        print(f'SKIP {mname}: checkpoint not found at {ckpt}')
        continue
    print(f'\n=== {mname} ===')
    model = load_model_from_checkpoint(ckpt)
    model.eval()

    for strat in ['greedy', 'beam', 'nucleus']:
        hyps, refs_l, hd = [], [], {}
        t0 = time.time()
        for i, iid in enumerate(eval_ids):
            cap = model.generate(
                val_embeddings[iid].unsqueeze(0),
                strategy=strat,
                num_beams=5,
                temperature=0.5,
            )
            hyps.append(cap)
            refs_l.append(val_refs[iid])
            hd[iid] = cap
            if (i + 1) % 100 == 0:
                print(f'  {i+1}/{len(eval_ids)} captions generated...')

        s = score_corpus(hyps, refs_l)
        c = compute_cider(hd, {k: val_refs[k] for k in eval_ids})
        if c is not None:
            s['CIDEr'] = c
        key = f'{mname}/{strat}'
        METRICS_RESULTS[key] = s
        print(
            f'  {key}: '
            f'BLEU-1={s["BLEU-1"]} '
            f'BLEU-4={s["BLEU-4"]} '
            f'METEOR={s["METEOR"]} '
            f'CIDEr={s.get("CIDEr", "N/A")} '
            f'ROUGE-L={s["ROUGE-L"]} '
            f'({time.time() - t0:.0f}s)'
        )

    del model
    torch.cuda.empty_cache()

print('\n' + '=' * 62)
print('PASTE into demo/index.html: replace const METRICS = {...};')
print('=' * 62)
print('const METRICS = ' + json.dumps(METRICS_RESULTS) + ';')


=== Frozen ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  Frozen/greedy: BLEU-1=0.6707 BLEU-4=0.2589 METEOR=0.4378 CIDEr=0.8764 ROUGE-L=0.5404 (176s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  Frozen/beam: BLEU-1=0.7049 BLEU-4=0.2896 METEOR=0.4619 CIDEr=0.9364 ROUGE-L=0.5552 (197s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  Frozen/nucleus: BLEU-1=0.6567 BLEU-4=0.2316 METEOR=0.4253 CIDEr=0.7892 ROUGE-L=0.5201 (185s)

=== FT-4L ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  FT-4L/greedy: BLEU-1=0.7182 BLEU-4=0.288 METEOR=0.4648 CIDEr=0.9001 ROUGE-L=0.5426 (179s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  FT-4L/beam: BLEU-1=0.7035 BLEU-4=0.2917 METEOR=0.4738 CIDEr=0.9155 ROUGE-L=0.5368 (196s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  FT-4L/nucleus: BLEU-1=0.7096 BLEU-4=0.269 METEOR=0.4568 CIDEr=0.8714 ROUGE-L=0.5315 (187s)

=== LoRA ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  LoRA/greedy: BLEU-1=0.5827 BLEU-4=0.1536 METEOR=0.3563 CIDEr=0.4935 ROUGE-L=0.4294 (175s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  LoRA/beam: BLEU-1=0.5828 BLEU-4=0.1835 METEOR=0.3704 CIDEr=0.5556 ROUGE-L=0.4408 (196s)
  100/500 captions generated...
  200/500 captions generated...
  300/500 captions generated...
  400/500 captions generated...
  500/500 captions generated...
  LoRA/nucleus: BLEU-1=0.5343 BLEU-4=0.1136 METEOR=0.3265 CIDEr=0.3617 ROUGE-L=0.3882 (185s)

PASTE into demo/index.html: replace const METRICS = {...};
const METRICS = {"Frozen/greedy": {"BLEU-1": 0.6707, "BLEU-4": 0.2589, "METEOR": 0.4378, "ROUGE-L": 0.5404, "CIDEr": 0.8764}, "Frozen/beam": {"BLEU-1": 0.7049, "BLEU-4": 0.2896, "METEOR

## Cell 18 — Sensitivity Sweep: BEAM_DATA / TEMP_DATA

- Sweeps beam width w=1,2,3,5,8,10 and temperature t=0.3 to 1.2 on LoRA.
- Produces BEAM_DATA and TEMP_DATA for Sensitivity tab charts.

In [30]:
SWEEP_N    = 200
sweep_ids  = eval_ids[:SWEEP_N]
sweep_embs = {iid: val_embeddings[iid] for iid in sweep_ids}
sweep_refs = {iid: val_refs[iid]       for iid in sweep_ids}

print(f'Loading LoRA checkpoint for sensitivity sweep ({SWEEP_N} images)...')
if not CKPT_LORA.exists():
    raise FileNotFoundError(f'LoRA checkpoint not found: {CKPT_LORA}')
m_sweep = load_model_from_checkpoint(CKPT_LORA)
m_sweep.eval()

def sweep_one(strategy, beam_width=5, temperature=0.5):
    hyps, refs_l, hd, bigrams = [], [], {}, []
    for iid in sweep_ids:
        cap = m_sweep.generate(
            sweep_embs[iid].unsqueeze(0),
            strategy=strategy,
            num_beams=beam_width,
            temperature=temperature,
        )
        hyps.append(cap)
        refs_l.append(sweep_refs[iid])
        hd[iid] = cap
        toks = nltk.word_tokenize(cap.lower())
        bigrams.extend(zip(toks, toks[1:]))
    s   = score_corpus(hyps, refs_l)
    cid = compute_cider(hd, {k: sweep_refs[k] for k in sweep_ids}) or 0.0
    d2  = round(len(set(bigrams)) / max(1, len(bigrams)), 4)
    return s['BLEU-4'], cid, d2

# Beam width sweep: strategy=beam, temperature fixed at 0.5
print('\nBeam width sweep [strategy=beam, temperature=0.5]')
BEAM_W = [1, 2, 3, 5, 8, 10]
bm_cider, bm_bleu4 = [], []
for w in BEAM_W:
    b4, cid, _ = sweep_one('beam', beam_width=w)
    bm_cider.append(round(cid, 4))
    bm_bleu4.append(round(b4, 4))
    print(f'  w={w:2d}:  BLEU-4={b4:.4f}  CIDEr={cid:.4f}')

# Temperature sweep: strategy=nucleus, beam width fixed at 5
print('\nTemperature sweep [strategy=nucleus, beam_width=5]')
TEMP_T = [0.3, 0.5, 0.7, 0.8, 1.0, 1.2]
tm_cider, tm_dist2 = [], []
for t in TEMP_T:
    b4, cid, d2 = sweep_one('nucleus', temperature=t)
    tm_cider.append(round(cid, 4))
    tm_dist2.append(d2)
    print(f'  t={t}:  CIDEr={cid:.4f}  Distinct-2={d2:.4f}')

del m_sweep
torch.cuda.empty_cache()

# Save all results to Drive
eval_out = {
    'METRICS':   METRICS_RESULTS,
    'BEAM_DATA': {'labels': BEAM_W,  'cider': bm_cider, 'bleu4': bm_bleu4},
    'TEMP_DATA': {'labels': TEMP_T,  'cider': tm_cider, 'distinct2': tm_dist2},
}
save_path = DRIVE_ROOT / 'eval_results_coco.json'
with open(save_path, 'w') as f:
    json.dump(eval_out, f, indent=2)
print(f'\nSaved to {save_path}')

print('\n' + '=' * 62)
print('PASTE into demo/index.html: replace const BEAM_DATA = {...};')
print('=' * 62)
print(f'const BEAM_DATA = {{\n  labels:{BEAM_W},\n  cider:{bm_cider},\n  bleu4:{bm_bleu4}\n}};')

print('\n' + '=' * 62)
print('PASTE into demo/index.html: replace const TEMP_DATA = {...};')
print('=' * 62)
print(f'const TEMP_DATA = {{\n  labels:{TEMP_T},\n  cider:{tm_cider},\n  distinct2:{tm_dist2}\n}};')

Loading LoRA checkpoint for sensitivity sweep (200 images)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Beam width sweep [strategy=beam, temperature=0.5]
  w= 1:  BLEU-4=0.1553  CIDEr=0.5109
  w= 2:  BLEU-4=0.1690  CIDEr=0.5638
  w= 3:  BLEU-4=0.1739  CIDEr=0.5861
  w= 5:  BLEU-4=0.1775  CIDEr=0.6099
  w= 8:  BLEU-4=0.1654  CIDEr=0.5718
  w=10:  BLEU-4=0.1639  CIDEr=0.5684

Temperature sweep [strategy=nucleus, beam_width=5]
  t=0.3:  CIDEr=0.4639  Distinct-2=0.5523
  t=0.5:  CIDEr=0.3751  Distinct-2=0.6248
  t=0.7:  CIDEr=0.2831  Distinct-2=0.7053
  t=0.8:  CIDEr=0.2290  Distinct-2=0.7869
  t=1.0:  CIDEr=0.0757  Distinct-2=0.8987


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


  t=1.2:  CIDEr=0.0282  Distinct-2=0.9699

Saved to /content/drive/MyDrive/IE7615_COCO_FULL/eval_results_coco.json

PASTE into demo/index.html: replace const BEAM_DATA = {...};
const BEAM_DATA = {
  labels:[1, 2, 3, 5, 8, 10],
  cider:[0.5109, 0.5638, 0.5861, 0.6099, 0.5718, 0.5684],
  bleu4:[0.1553, 0.169, 0.1739, 0.1775, 0.1654, 0.1639]
};

PASTE into demo/index.html: replace const TEMP_DATA = {...};
const TEMP_DATA = {
  labels:[0.3, 0.5, 0.7, 0.8, 1.0, 1.2],
  cider:[0.4639, 0.3751, 0.2831, 0.229, 0.0757, 0.0282],
  distinct2:[0.5523, 0.6248, 0.7053, 0.7869, 0.8987, 0.9699]
};


## Cell 19 — Print All Constants for demo/index.html

Loads eval_results_coco.json from Drive and prints all 3 copy-paste blocks.

After pasting into index.html, restart demo/run.sh to see updated results.

In [31]:
import json
from pathlib import Path

p = Path('/content/drive/MyDrive/IE7615_COCO_FULL/eval_results_coco.json')
if not p.exists():
    raise FileNotFoundError('Run Cell 17 and Cell 18 first: ' + str(p))

with open(p) as f:
    ev = json.load(f)

M  = ev['METRICS']
BD = ev['BEAM_DATA']
TD = ev['TEMP_DATA']
SEP = '=' * 62

print(SEP)
print('1. Replace in index.html:  const METRICS = {...};')
print(SEP)
print('const METRICS = ' + json.dumps(M) + ';')

print('')
print(SEP)
print('2. Replace in index.html:  const BEAM_DATA = {...};')
print(SEP)
print('const BEAM_DATA = {')
print(f'  labels:{BD["labels"]},')
print(f'  cider:{BD["cider"]},')
print(f'  bleu4:{BD["bleu4"]}')
print('};')

print('')
print(SEP)
print('3. Replace in index.html:  const TEMP_DATA = {...};')
print(SEP)
print('const TEMP_DATA = {')
print(f'  labels:{TD["labels"]},')
print(f'  cider:{TD["cider"]},')
print(f'  distinct2:{TD["distinct2"]}')
print('};')

1. Replace in index.html:  const METRICS = {...};
const METRICS = {"Frozen/greedy": {"BLEU-1": 0.6707, "BLEU-4": 0.2589, "METEOR": 0.4378, "ROUGE-L": 0.5404, "CIDEr": 0.8764}, "Frozen/beam": {"BLEU-1": 0.7049, "BLEU-4": 0.2896, "METEOR": 0.4619, "ROUGE-L": 0.5552, "CIDEr": 0.9364}, "Frozen/nucleus": {"BLEU-1": 0.6567, "BLEU-4": 0.2316, "METEOR": 0.4253, "ROUGE-L": 0.5201, "CIDEr": 0.7892}, "FT-4L/greedy": {"BLEU-1": 0.7182, "BLEU-4": 0.288, "METEOR": 0.4648, "ROUGE-L": 0.5426, "CIDEr": 0.9001}, "FT-4L/beam": {"BLEU-1": 0.7035, "BLEU-4": 0.2917, "METEOR": 0.4738, "ROUGE-L": 0.5368, "CIDEr": 0.9155}, "FT-4L/nucleus": {"BLEU-1": 0.7096, "BLEU-4": 0.269, "METEOR": 0.4568, "ROUGE-L": 0.5315, "CIDEr": 0.8714}, "LoRA/greedy": {"BLEU-1": 0.5827, "BLEU-4": 0.1536, "METEOR": 0.3563, "ROUGE-L": 0.4294, "CIDEr": 0.4935}, "LoRA/beam": {"BLEU-1": 0.5828, "BLEU-4": 0.1835, "METEOR": 0.3704, "ROUGE-L": 0.4408, "CIDEr": 0.5556}, "LoRA/nucleus": {"BLEU-1": 0.5343, "BLEU-4": 0.1136, "METEOR": 0.3265, "RO